# Shark Tank Investor Simulator

Run the installation cell, then the Groq setup/test cell, then the application cell. Your API key is entered privately and stored locally outside the notebook.


In [ ]:
%pip install customtkinter groq python-dotenv


In [ ]:
# Step 1: Import required libraries
import os
from pathlib import Path
from getpass import getpass
from groq import Groq

# Step 2: Enter your Groq API key securely
# This key is saved on your computer, NOT inside the notebook.
api_key = getpass("Enter your Groq API Key: ").strip()
if not api_key:
    raise ValueError("A Groq API key is required.")

# Step 3: Initialize Groq client
client = Groq(api_key=api_key)

# Step 4: Select the working model
MODEL = "openai/gpt-oss-120b"
print("Groq API client initialized!")
print("Selected Model:", MODEL)

# Step 5: Create AI response function
def ai(system, user, tokens=500, temperature=0.65):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        max_completion_tokens=max(tokens, 1024),
        temperature=temperature,
        reasoning_effort="low",
    )
    return response.choices[0].message.content or ""

# Step 6: Test the API connection
try:
    response = ai(
        "You are a helpful assistant.",
        "Say Shark Tank is ready!",
        tokens=100,
    )
    print("\nAPI Test Result:")
    print(response)

    # Save the key in your home directory so the desktop app can load it.
    # Never upload this .env file to GitHub.
    env_file = Path.home() / ".shark_tank.env"
    env_file.write_text(f"GROQ_API_KEY={api_key}\n", encoding="utf-8")
    print("API key saved locally for the Shark Tank app.")
except Exception as e:
    print("\nAPI Error:", e)
    print("Check your API key and model before launching the app.")


In [ ]:
import os
import json
import re
import threading
import queue
import random
from pathlib import Path
from datetime import datetime
from tkinter import messagebox, filedialog

import customtkinter as ctk
from groq import Groq
from dotenv import dotenv_values

MODEL = "openai/gpt-oss-120b"
ENV_FILE = Path.home() / ".shark_tank.env"

BG = "#071421"
PANEL = "#10273B"
CARD = "#142F46"
GOLD = "#F0B957"
WHITE = "#F4F7FA"
MUTED = "#B4C7D7"
TEAL = "#2BC3A5"
RED = "#E77777"

ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")


SHARKS = {
    "Kevin O'Leary": {
        "focus": "Finance and Valuation",
        "intro": "Examines profitability, valuation, cash flow, margins and investment returns.",
        "style": "Skeptical, direct and numbers-focused."
    },
    "Mark Cuban": {
        "focus": "Technology and Scalability",
        "intro": "Examines innovation, competition, scalability, founders and execution.",
        "style": "Analytical, direct and commercially demanding."
    },
    "Barbara Corcoran": {
        "focus": "Real Estate and Sales",
        "intro": "Focuses on sales, customer demand, resilience and founder potential.",
        "style": "Intuitive, candid and practical."
    },
    "Lori Greiner": {
        "focus": "Products and Distribution",
        "intro": "Examines product-market fit, usability, customer needs and distribution.",
        "style": "Practical, observant and constructive."
    },
    "Robert Herjavec": {
        "focus": "Technology and Operations",
        "intro": "Focuses on operational efficiency, execution, technology and leadership.",
        "style": "Friendly, analytical and execution-focused."
    },
    "Daymond John": {
        "focus": "Branding and Marketing",
        "intro": "Examines branding, marketing, customer acquisition and capital efficiency.",
        "style": "Resource-conscious, entrepreneurial and direct."
    },
    "Pratham Mittal": {
        "focus": "Entrepreneurship and Validation",
        "intro": "Focuses on experimentation, customer validation and sustainable growth.",
        "style": "Curious, thoughtful and commercially practical."
    }
}

def label(parent, text, size=13, color=WHITE, bold=False):
    widget = ctk.CTkLabel(
        parent,
        text=str(text),
        text_color=color,
        font=("Segoe UI", size, "bold" if bold else "normal"),
        justify="left",
        anchor="w",
        wraplength=850
    )
    widget.pack(anchor="w", pady=4)
    return widget


def button(parent, text, command, gold=False):
    return ctk.CTkButton(
        parent,
        text=text,
        command=command,
        height=40,
        corner_radius=9,
        fg_color=GOLD if gold else "#244A66",
        hover_color="#D8A23E" if gold else "#346381",
        text_color=BG if gold else WHITE,
        font=("Segoe UI", 12, "bold")
    )


def field(parent, title, default="", multiline=False, height=110):
    label(parent, title, 12, MUTED, True)

    if multiline:
        widget = ctk.CTkTextbox(
            parent,
            height=height,
            fg_color=CARD,
            text_color=WHITE,
            wrap="word",
            border_width=1,
            border_color="#31516A"
        )
        widget.pack(fill="x", pady=(0, 10))
        if default:
            widget.insert("1.0", str(default))
    else:
        widget = ctk.CTkEntry(
            parent,
            height=38,
            fg_color=CARD,
            text_color=WHITE,
            border_color="#31516A"
        )
        widget.pack(fill="x", pady=(0, 10))
        if default != "":
            widget.insert(0, str(default))

    return widget


def value(widget):
    if isinstance(widget, ctk.CTkTextbox):
        return widget.get("1.0", "end").strip()
    return widget.get().strip()


def write_text(widget, text):
    widget.configure(state="normal")
    widget.delete("1.0", "end")
    widget.insert("1.0", str(text))
    widget.configure(state="disabled")


def clean_markdown(text):
    """Make AI-generated Markdown readable in a plain textbox."""
    lines = []

    for original in str(text).splitlines():
        line = original.strip()

        if re.fullmatch(r"[-*_]{3,}", line):
            lines.append("")
            continue

        line = re.sub(r"^#{1,6}\s*", "", line)
        line = line.replace("**", "")
        line = re.sub(r"(?<!\w)\*(?=\S)", "", line)

        if line.startswith("|") and line.endswith("|"):
            cells = [c.strip() for c in line.strip("|").split("|")]

            if cells and all(
                re.fullmatch(r":?-{3,}:?", c)
                for c in cells
            ):
                continue

            line = "  |  ".join(cells)

        lines.append(line)

    return "\n".join(lines)


def parse_json_response(text):
    """Extract JSON even if the model includes code fences."""
    cleaned = str(text).strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        start = cleaned.find("{")
        end = cleaned.rfind("}")

        if start >= 0 and end > start:
            return json.loads(cleaned[start:end + 1])

        raise

class SharkTankApp(ctk.CTk):

    def __init__(self):
        super().__init__()

        self.title("SHARK TANK | Investor Arena")
        self.geometry("1250x820")
        self.minsize(980, 650)
        self.configure(fg_color=BG)

        saved_key = dotenv_values(ENV_FILE).get("GROQ_API_KEY")
        saved_key = saved_key or os.getenv("GROQ_API_KEY")

        self.client = Groq(api_key=saved_key) if saved_key else None
        self.model = MODEL

        self.startup = {}
        self.pitch = ""
        self.panel = []
        self.rounds = []
        self.current = 0
        self.evaluations = {}
        self.negotiations = []
        self.report = ""

        self.busy = False
        self.results = queue.Queue()
        self.current_page = "Dashboard"

        self.build_layout()
        self.show_page("Dashboard")
        self.after(100, self.process_results)

    def ask_ai(self, system, prompt, tokens=900):
        if self.client is None:
            raise RuntimeError(
                "Groq API key not configured. Open Settings."
            )

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5,
            max_completion_tokens=tokens,
            reasoning_effort="low"
        )

        content = response.choices[0].message.content

        if not content or not content.strip():
            raise RuntimeError(
                "Groq returned an empty response. Try again."
            )

        return content.strip()

    def run_ai(self, system, prompt, callback, tokens=900):
        if self.busy:
            messagebox.showinfo(
                "Please wait",
                "An investor response is already generating."
            )
            return

        if self.client is None:
            messagebox.showwarning(
                "Groq not configured",
                "Open Settings and enter your API key."
            )
            return

        self.busy = True
        self.set_status("● Generating response...", GOLD)

        def worker():
            try:
                result = self.ask_ai(system, prompt, tokens)
                self.results.put((callback, result, None))
            except Exception as exc:
                self.results.put((callback, None, str(exc)))

        threading.Thread(target=worker, daemon=True).start()

    def process_results(self):
        try:
            while True:
                callback, result, error = self.results.get_nowait()

                self.busy = False

                self.set_status(
                    "● Groq configured" if self.client
                    else "● API not connected",
                    TEAL if self.client else RED
                )

                if error:
                    messagebox.showerror("Groq API Error", error)
                else:
                    try:
                        callback(result)
                    except Exception as exc:
                        messagebox.showerror(
                            "Application Error", str(exc)
                        )

        except queue.Empty:
            pass

        if self.winfo_exists():
            self.after(100, self.process_results)

    def set_status(self, text, color=TEAL):
        self.status.configure(text=text, text_color=color)

    def investor_prompt(self, name):
        info = SHARKS[name]

        return f"""
You are a fictional investor persona inspired by the public
business approach of {name}.

Specialization: {info['focus']}
Style: {info['style']}

Ask realistic, specific questions.
Be skeptical about unsupported numbers.
Avoid repetitive questions.
Use the entrepreneur's actual statements.
Do not invent startup traction or achievements.

Do not claim to be the real investor.
All investment offers are fictional educational simulations.
"""

    def build_layout(self):
        self.sidebar = ctk.CTkFrame(
            self,
            width=225,
            fg_color=PANEL,
            corner_radius=0
        )
        self.sidebar.pack(side="left", fill="y")
        self.sidebar.pack_propagate(False)

        ctk.CTkLabel(
            self.sidebar,
            text="🦈 SHARK TANK",
            text_color=GOLD,
            font=("Segoe UI", 23, "bold")
        ).pack(pady=(25, 5))

        ctk.CTkLabel(
            self.sidebar,
            text="INVESTOR ARENA",
            text_color=MUTED,
            font=("Segoe UI", 10, "bold")
        ).pack(pady=(0, 20))

        self.pages = [
            "Dashboard",
            "Startup",
            "Pitch",
            "Investors",
            "Q&A",
            "Evaluation",
            "Negotiation",
            "Report",
            "Settings"
        ]

        for page in self.pages:
            button(
                self.sidebar,
                page,
                lambda p=page: self.show_page(p)
            ).pack(fill="x", padx=14, pady=4)

        label(
            self.sidebar,
            "SHARK TANK • PITCH TO WIN",
            10,
            MUTED
        ).pack(side="bottom", pady=18)

        self.main = ctk.CTkFrame(
            self,
            fg_color=BG,
            corner_radius=0
        )
        self.main.pack(side="left", fill="both", expand=True)

        self.header = ctk.CTkFrame(
            self.main,
            fg_color=PANEL,
            height=80
        )
        self.header.pack(fill="x", padx=18, pady=(18, 8))
        self.header.pack_propagate(False)

        ctk.CTkLabel(
            self.header,
            text="SHARK TANK",
            text_color=GOLD,
            font=("Segoe UI", 30, "bold")
        ).pack(side="left", padx=24)

        self.status = ctk.CTkLabel(
            self.header,
            text=(
                "● Groq configured"
                if self.client
                else "● API not connected"
            ),
            text_color=TEAL if self.client else RED,
            font=("Segoe UI", 12, "bold")
        )
        self.status.pack(side="right", padx=20)

        self.content = ctk.CTkScrollableFrame(
            self.main,
            fg_color=BG
        )
        self.content.pack(
            fill="both",
            expand=True,
            padx=18,
            pady=10
        )

    def clear_page(self, heading, subtitle=""):
        for widget in self.content.winfo_children():
            widget.destroy()

        label(self.content, heading, 27, GOLD, True)

        if subtitle:
            label(self.content, subtitle, 13, MUTED)

    def show_page(self, page):
        if self.busy:
            return
        self.current_page = page

        routes = {
            "Dashboard": self.dashboard,
            "Startup": self.startup_page,
            "Pitch": self.pitch_page,
            "Investors": self.investors_page,
            "Q&A": self.qa_page,
            "Evaluation": self.evaluation_page,
            "Negotiation": self.negotiation_page,
            "Report": self.report_page,
            "Settings": self.settings_page
        }

        routes[page]()

    def dashboard(self):
        self.clear_page(
            "WELCOME TO THE INVESTOR ARENA",
            "Pitch • Defend • Negotiate • Grow"
        )

        stats = ctk.CTkFrame(
            self.content,
            fg_color=PANEL
        )
        stats.pack(fill="x", pady=15)

        items = [
            ("STARTUP", self.startup.get("name", "Not registered")),
            (
                "FUNDING ASK",
                f"₹{self.startup.get('investment', 0):,.0f}"
                if self.startup else "—"
            ),
            (
                "EQUITY",
                f"{self.startup.get('equity', 0):g}%"
                if self.startup else "—"
            ),
            ("QUESTIONS", f"{len(self.rounds)}/5")
        ]

        for title, data in items:
            card = ctk.CTkFrame(stats, fg_color=CARD)
            card.pack(
                side="left",
                expand=True,
                fill="both",
                padx=6,
                pady=10
            )

            ctk.CTkLabel(
                card,
                text=title,
                text_color=MUTED
            ).pack(pady=(12, 4))

            ctk.CTkLabel(
                card,
                text=str(data),
                text_color=GOLD,
                font=("Segoe UI", 17, "bold")
            ).pack(pady=(0, 12))

        label(self.content, "MEET THE SHARKS", 21, WHITE, True)

        for name, info in SHARKS.items():
            card = ctk.CTkFrame(
                self.content,
                fg_color=PANEL,
                corner_radius=12
            )
            card.pack(fill="x", pady=5)

            label(card, "🦈 " + name, 17, GOLD, True)
            label(card, info["focus"], 12, TEAL, True)
            label(card, info["intro"], 12, MUTED)

        label(
            self.content,
            "All investor personas and investment offers are fictional.",
            11,
            MUTED
        )

        controls = ctk.CTkFrame(
            self.content,
            fg_color="transparent"
        )
        controls.pack(anchor="w", pady=15)

        button(
            controls,
            "Save Session",
            self.save_session,
            True
        ).pack(side="left", padx=(0, 10))

        button(
            controls,
            "Load Session",
            self.load_session
        ).pack(side="left")

    def save_session(self):
        data = {
            "version": 1,
            "startup": self.startup,
            "pitch": self.pitch,
            "panel": self.panel,
            "rounds": self.rounds,
            "current": self.current,
            "evaluations": self.evaluations,
            "negotiations": self.negotiations,
            "report": self.report
        }

        path = filedialog.asksaveasfilename(
            title="Save Shark Tank Session",
            defaultextension=".json",
            initialfile="shark_tank_session.json",
            filetypes=[("JSON Files", "*.json")]
        )

        if not path:
            return

        Path(path).write_text(
            json.dumps(data, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

        messagebox.showinfo(
            "Saved",
            "Your Shark Tank session was saved successfully."
        )

    def load_session(self):
        path = filedialog.askopenfilename(
            title="Load Shark Tank Session",
            filetypes=[("JSON Files", "*.json")]
        )

        if not path:
            return

        try:
            data = json.loads(
                Path(path).read_text(encoding="utf-8")
            )

            startup = data.get("startup", {})
            panel = data.get("panel", [])
            rounds = data.get("rounds", [])
            evaluations = data.get("evaluations", {})

            if not isinstance(startup, dict):
                raise ValueError("Invalid startup data.")

            if not isinstance(panel, list):
                raise ValueError("Invalid investor panel.")

            if not isinstance(rounds, list):
                raise ValueError("Invalid Q&A data.")

            if not isinstance(evaluations, dict):
                raise ValueError("Invalid evaluation data.")

            if any(name not in SHARKS for name in panel):
                raise ValueError("Unknown investor in saved panel.")

            self.startup = startup
            self.pitch = str(data.get("pitch", ""))
            self.panel = panel
            self.rounds = rounds
            self.current = max(
                0,
                min(int(data.get("current", 0)), len(panel))
            )
            self.evaluations = evaluations
            self.negotiations = data.get("negotiations", [])
            self.report = str(data.get("report", ""))

            self.show_page("Dashboard")

            messagebox.showinfo(
                "Session Loaded",
                "Your saved simulation has been restored."
            )

        except Exception as exc:
            messagebox.showerror(
                "Load Error",
                f"Could not load session:\n{exc}"
            )

    def settings_page(self):
        self.clear_page(
            "API SETTINGS",
            "Your Groq API key loads automatically from your saved file."
        )

        label(
            self.content,
            "Saved API key: " + (
                "Found" if self.client else "Not found"
            ),
            14,
            TEAL if self.client else RED,
            True
        )

        self.key_entry = field(
            self.content,
            "New Groq API Key (optional)"
        )
        self.key_entry.configure(show="*")

        self.model_entry = field(
            self.content,
            "Groq Model",
            self.model
        )

        def connect():
            new_key = self.key_entry.get().strip()
            new_model = self.model_entry.get().strip()

            if not new_key and not self.client:
                messagebox.showwarning(
                    "Missing key",
                    "Enter your Groq API key."
                )
                return

            old_client = self.client
            old_model = self.model

            if new_key:
                self.client = Groq(api_key=new_key)

            self.model = new_model or MODEL

            def success(result):
                if new_key:
                    ENV_FILE.write_text(
                        f"GROQ_API_KEY={new_key}\n",
                        encoding="utf-8"
                    )

                self.set_status("● Groq connected", TEAL)

                messagebox.showinfo(
                    "Connection Successful",
                    result
                )

            # Test the selected connection
            self.run_ai(
                "You are a helpful assistant.",
                "Reply exactly: Shark Tank is ready!",
                success,
                200
            )

        button(
            self.content,
            "Connect & Test",
            connect,
            True
        ).pack(anchor="w", pady=15)

    def startup_page(self):
        self.clear_page(
            "REGISTER YOUR STARTUP",
            "Enter your business idea and funding requirements."
        )

        defaults = [
            ("name", "Startup Name", "EcoStruct"),
            ("industry", "Industry", "Sustainable Construction"),
            (
                "idea",
                "Business Idea",
                "Affordable, energy-efficient homes using sustainable "
                "materials and smart construction technologies."
            ),
            (
                "customers",
                "Target Customers",
                "Homeowners and real estate developers"
            ),
            (
                "business_model",
                "Revenue Model",
                "Construction services, design and partnerships"
            ),
            ("revenue", "Current Revenue", "Pre-revenue"),
            ("investment", "Investment Requested (₹)", "5000000"),
            ("equity", "Equity Offered (%)", "10")
        ]

        self.startup_fields = {}

        for key, title, default in defaults:
            self.startup_fields[key] = field(
                self.content,
                title,
                self.startup.get(key, default),
                multiline=key in (
                    "idea", "customers", "business_model"
                )
            )

        def register():
            data = {
                key: value(widget)
                for key, widget in self.startup_fields.items()
            }

            try:
                data["investment"] = float(data["investment"])
                data["equity"] = float(data["equity"])

                if not data["name"] or not data["idea"]:
                    raise ValueError()

                if data["investment"] <= 0:
                    raise ValueError()

                if not 0 < data["equity"] <= 100:
                    raise ValueError()

            except ValueError:
                messagebox.showerror(
                    "Invalid Input",
                    "Check the startup name, business idea, "
                    "investment amount and equity percentage."
                )
                return

            self.startup = data
            self.pitch = ""
            self.panel = []
            self.rounds = []
            self.current = 0
            self.evaluations = {}
            self.negotiations = []
            self.report = ""

            valuation = (
                data["investment"] / (data["equity"] / 100)
            )

            messagebox.showinfo(
                "Startup Registered",
                f"{data['name']} registered successfully!\n\n"
                f"Implied post-money valuation: ₹{valuation:,.0f}"
            )

            self.show_page("Pitch")

        button(
            self.content,
            "Register Startup",
            register,
            True
        ).pack(anchor="w", pady=15)

    def pitch_page(self):
        self.clear_page(
            "PRESENT YOUR PITCH",
            "Prepare a persuasive startup pitch."
        )

        self.pitch_editor = field(
            self.content,
            "Your Startup Pitch",
            self.pitch,
            multiline=True,
            height=260
        )

        def generate():
            if not self.startup:
                messagebox.showwarning(
                    "Startup Required",
                    "Register your startup first."
                )
                return

            def receive(text):
                self.pitch_editor.delete("1.0", "end")
                self.pitch_editor.insert("1.0", text)

            self.run_ai(
                "You are an experienced startup pitch coach. "
                "Write a compelling 90-second investor pitch. "
                "Use only supplied facts. Do not invent traction, "
                "revenue or achievements. Use natural spoken language.",
                json.dumps(self.startup),
                receive,
                1200
            )

        def save():
            pitch = value(self.pitch_editor)

            if not pitch:
                messagebox.showwarning(
                    "Missing Pitch",
                    "Write or generate a pitch first."
                )
                return

            self.pitch = pitch
            self.rounds = []
            self.current = 0
            self.evaluations = {}
            self.negotiations = []
            self.report = ""

            messagebox.showinfo(
                "Pitch Saved",
                "Your pitch is ready for the Sharks."
            )

            self.show_page("Investors")

        controls = ctk.CTkFrame(
            self.content,
            fg_color="transparent"
        )
        controls.pack(anchor="w", pady=15)

        button(
            controls,
            "Generate with AI",
            generate
        ).pack(side="left", padx=(0, 10))

        button(
            controls,
            "Save Pitch",
            save,
            True
        ).pack(side="left")

    def investors_page(self):
        self.clear_page(
            "SELECT YOUR SHARKS",
            "Choose exactly five investors for your pitch."
        )

        self.shark_vars = {}

        for name, info in SHARKS.items():
            var = ctk.BooleanVar(value=name in self.panel)
            self.shark_vars[name] = var

            ctk.CTkCheckBox(
                self.content,
                text=f"{name} — {info['focus']}",
                variable=var,
                text_color=WHITE
            ).pack(anchor="w", pady=9)

        def choose_random():
            selected = random.sample(list(SHARKS), 5)

            for name, var in self.shark_vars.items():
                var.set(name in selected)

        def confirm():
            selected = [
                name
                for name, var in self.shark_vars.items()
                if var.get()
            ]

            if len(selected) != 5:
                messagebox.showwarning(
                    "Select Five",
                    "You must select exactly five Sharks."
                )
                return

            if not self.startup or not self.pitch:
                messagebox.showwarning(
                    "Pitch Required",
                    "Register your startup and save a pitch first."
                )
                return

            self.panel = selected
            self.rounds = []
            self.current = 0
            self.evaluations = {}
            self.negotiations = []
            self.report = ""

            self.show_page("Q&A")

        button(
            self.content,
            "Select Random Five",
            choose_random
        ).pack(anchor="w", pady=8)

        button(
            self.content,
            "Confirm Panel",
            confirm,
            True
        ).pack(anchor="w", pady=8)

    def qa_page(self):
        self.clear_page(
            "THE INVESTOR HOT SEAT",
            "Answer challenging questions from five investors."
        )

        if not self.startup or not self.pitch or len(self.panel) != 5:
            label(
                self.content,
                "Register your startup, save your pitch, "
                "and select five investors first.",
                14,
                MUTED
            )
            return

        if self.current >= len(self.panel):
            label(
                self.content,
                "All five investor rounds are complete!",
                20,
                TEAL,
                True
            )

            button(
                self.content,
                "Proceed to Evaluation",
                lambda: self.show_page("Evaluation"),
                True
            ).pack(anchor="w", pady=15)
            return

        name = self.panel[self.current]

        label(
            self.content,
            f"INVESTOR {self.current + 1}/5 — {name}",
            20,
            GOLD,
            True
        )

        if len(self.rounds) <= self.current:
            label(
                self.content,
                "The investor is ready to challenge your business.",
                13,
                MUTED
            )

            button(
                self.content,
                "Generate Investor Question",
                self.generate_question,
                True
            ).pack(anchor="w", pady=15)
            return

        item = self.rounds[self.current]

        label(
            self.content,
            clean_markdown(item["question"]),
            16,
            WHITE,
            True
        )

        self.answer_editor = field(
            self.content,
            "Your Answer",
            item.get("answer", ""),
            multiline=True,
            height=160
        )

        if item.get("followup"):
            label(
                self.content,
                "FOLLOW-UP QUESTION",
                17,
                GOLD,
                True
            )

            label(
                self.content,
                clean_markdown(item["followup"]),
                15,
                WHITE
            )

            self.followup_editor = field(
                self.content,
                "Your Follow-up Answer",
                item.get("followup_answer", ""),
                multiline=True,
                height=130
            )
        else:
            self.followup_editor = None

        def save_answer():
            answer = value(self.answer_editor)

            if not answer:
                messagebox.showwarning(
                    "Missing Answer",
                    "Please answer the investor's question."
                )
                return

            item["answer"] = answer

            if self.followup_editor is not None:
                item["followup_answer"] = value(
                    self.followup_editor
                )

            messagebox.showinfo(
                "Saved",
                "Your response has been saved."
            )

        def request_followup():
            answer = value(self.answer_editor)

            if not answer:
                messagebox.showwarning(
                    "Answer Required",
                    "Answer the first question before requesting a follow-up."
                )
                return

            item["answer"] = answer

            if item.get("followup"):
                messagebox.showinfo(
                    "Already Generated",
                    "This investor has already asked a follow-up."
                )
                return

            prompt = (
                f"Startup:\n{json.dumps(self.startup)}\n\n"
                f"Original question:\n{item['question']}\n\n"
                f"Entrepreneur answer:\n{answer}\n\n"
                "Ask ONE specific challenging follow-up question. "
                "Do not repeat the original question. "
                "Return only the follow-up question."
            )

            def receive(text):
                item["followup"] = text
                self.after(50, lambda: self.show_page("Q&A"))

            self.run_ai(
                self.investor_prompt(name),
                prompt,
                receive,
                550
            )

        def next_investor():
            answer = value(self.answer_editor)

            if not answer:
                messagebox.showwarning(
                    "Answer Required",
                    "Answer the investor before continuing."
                )
                return

            item["answer"] = answer

            if item.get("followup"):
                followup_answer = (
                    value(self.followup_editor)
                    if self.followup_editor is not None
                    else item.get("followup_answer", "")
                )

                if not followup_answer:
                    messagebox.showwarning(
                        "Follow-up Required",
                        "Answer the follow-up before continuing."
                    )
                    return

                item["followup_answer"] = followup_answer

            self.current += 1
            self.after(50, lambda: self.show_page("Q&A"))

        controls = ctk.CTkFrame(
            self.content,
            fg_color="transparent"
        )
        controls.pack(anchor="w", pady=12)

        button(
            controls,
            "Save Answer",
            save_answer
        ).pack(side="left", padx=(0, 10))

        button(
            controls,
            "Request Follow-up",
            request_followup
        ).pack(side="left", padx=(0, 10))

        button(
            controls,
            "Next Investor →",
            next_investor,
            True
        ).pack(side="left")

    def generate_question(self):
        if self.current >= len(self.panel):
            return

        name = self.panel[self.current]

        previous = [
            round_data["question"]
            for round_data in self.rounds
        ]

        prompt = (
            f"Startup details:\n{json.dumps(self.startup)}\n\n"
            f"Pitch:\n{self.pitch}\n\n"
            f"Previous questions:\n{json.dumps(previous)}\n\n"
            "Ask exactly ONE realistic, challenging investor question "
            "specific to this business. Focus on your investment expertise. "
            "Avoid repeating earlier questions. "
            "Use the actual funding ask and equity from the startup data. "
            "Do not invent contradictory investment figures. "
            "Return only the question, without Markdown headings."
        )

        def receive(question):
            self.rounds.append({
                "shark": name,
                "question": question,
                "answer": "",
                "followup": "",
                "followup_answer": ""
            })

            self.after(50, lambda: self.show_page("Q&A"))

        self.run_ai(
            self.investor_prompt(name),
            prompt,
            receive,
            600
        )

    def evaluation_page(self):
        self.clear_page(
            "INVESTOR EVALUATIONS",
            "Detailed feedback, scores and fictional offers."
        )

        if self.current < 5 or len(self.rounds) != 5:
            label(
                self.content,
                "Complete all five investor rounds first.",
                14,
                MUTED
            )
            return

        if len(self.evaluations) < 5:
            label(
                self.content,
                f"Evaluated {len(self.evaluations)} of 5 investors.",
                13,
                MUTED
            )

            button(
                self.content,
                "Evaluate My Pitch",
                self.evaluate_next,
                True
            ).pack(anchor="w", pady=12)

        for name, data in self.evaluations.items():
            card = ctk.CTkFrame(
                self.content,
                fg_color=PANEL,
                corner_radius=12
            )
            card.pack(fill="x", pady=8)

            label(
                card,
                f"{name} — {data.get('score', 0)}/10",
                19,
                GOLD,
                True
            )

            for key in (
                "strengths",
                "weaknesses",
                "concern",
                "reasoning"
            ):
                label(
                    card,
                    f"{key.title()}: {data.get(key, '')}",
                    12,
                    WHITE
                )

            decision = data.get("decision", "OUT")

            label(
                card,
                f"Decision: {decision}",
                14,
                TEAL if decision != "OUT" else RED,
                True
            )

            if decision != "OUT":
                amount = data.get("amount_inr") or 0
                equity = data.get("equity_percent") or 0

                label(
                    card,
                    f"Fictional offer: ₹{amount:,.0f} "
                    f"for {equity:g}% equity",
                    13,
                    GOLD,
                    True
                )

        if len(self.evaluations) == 5:
            total = sum(
                data.get("score", 0)
                for data in self.evaluations.values()
            )

            label(
                self.content,
                f"OVERALL SCORE: {total}/50",
                24,
                GOLD,
                True
            )

            button(
                self.content,
                "Proceed to Negotiation",
                lambda: self.show_page("Negotiation"),
                True
            ).pack(anchor="w", pady=15)

    def evaluate_next(self):
        remaining = [
            item for item in self.rounds
            if item["shark"] not in self.evaluations
        ]

        if not remaining:
            self.show_page("Evaluation")
            return

        item = remaining[0]
        name = item["shark"]

        system = self.investor_prompt(name) + """
Evaluate the entrepreneur's business and answers realistically.

Return ONLY a valid JSON object:
{
  "score": 0,
  "strengths": "specific strengths",
  "weaknesses": "specific weaknesses",
  "concern": "biggest investment concern",
  "decision": "OUT",
  "amount_inr": null,
  "equity_percent": null,
  "reasoning": "brief explanation"
}

Rules:
- Score must be an integer from 0 to 10.
- Decision must be INVEST, COUNTEROFFER, or OUT.
- If OUT, amount_inr and equity_percent must be null.
- If investing, specify a realistic positive amount and equity.
- Evaluate the evidence rather than simply rewarding confidence.
- Do not invent achievements, contracts or financial results.
- Offers are fictional.
"""

        prompt = (
            f"Startup:\n{json.dumps(self.startup)}\n\n"
            f"Pitch:\n{self.pitch}\n\n"
            f"Investor Q&A:\n{json.dumps(item)}"
        )

        def receive(text):
            try:
                data = parse_json_response(text)

                data["score"] = max(
                    0,
                    min(10, int(data.get("score", 0)))
                )

                decision = str(
                    data.get("decision", "OUT")
                ).upper()

                if decision not in (
                    "INVEST", "COUNTEROFFER", "OUT"
                ):
                    decision = "OUT"

                if decision != "OUT":
                    try:
                        amount = float(data["amount_inr"])
                        equity = float(data["equity_percent"])

                        if amount <= 0 or not 0 < equity <= 100:
                            raise ValueError()

                        data["amount_inr"] = amount
                        data["equity_percent"] = equity

                    except (ValueError, TypeError, KeyError):
                        decision = "OUT"

                if decision == "OUT":
                    data["amount_inr"] = None
                    data["equity_percent"] = None

                data["decision"] = decision

                self.evaluations[name] = data
                self.report = ""

                if len(self.evaluations) < 5:
                    self.after(100, self.evaluate_next)
                else:
                    self.after(
                        100,
                        lambda: self.show_page("Evaluation")
                    )

            except Exception as exc:
                messagebox.showerror(
                    "Evaluation Error",
                    f"Could not read the investor evaluation:\n{exc}\n\n"
                    "Please click Evaluate My Pitch again."
                )
                self.show_page("Evaluation")

        self.run_ai(
            system,
            prompt,
            receive,
            1400
        )

    def negotiation_page(self):
        self.clear_page(
            "DEAL NEGOTIATION",
            "Negotiate with interested fictional investor personas."
        )

        available = [
            name
            for name, data in self.evaluations.items()
            if data.get("decision") in (
                "INVEST", "COUNTEROFFER"
            )
        ]

        if not available:
            label(
                self.content,
                "There are currently no investment offers. "
                "Complete the evaluation to see investor decisions.",
                14,
                MUTED
            )

            if len(self.evaluations) == 5:
                button(
                    self.content,
                    "View Final Report",
                    lambda: self.show_page("Report"),
                    True
                ).pack(anchor="w", pady=12)

            return

        self.shark_selector = ctk.CTkOptionMenu(
            self.content,
            values=available,
            fg_color="#244A66",
            button_color="#346381"
        )
        self.shark_selector.pack(anchor="w", pady=12)

        self.counter_editor = field(
            self.content,
            "Your Counteroffer",
            multiline=True,
            height=120
        )

        self.negotiation_result = ctk.CTkTextbox(
            self.content,
            height=270,
            fg_color=CARD,
            text_color=WHITE,
            font=("Segoe UI", 14),
            wrap="word"
        )
        self.negotiation_result.pack(fill="x", pady=15)

        def negotiate():
            name = self.shark_selector.get()
            counteroffer = value(self.counter_editor)

            if not counteroffer:
                messagebox.showwarning(
                    "Counteroffer Required",
                    "Enter your proposed deal terms."
                )
                return

            history = [
                entry
                for entry in self.negotiations
                if entry["shark"] == name
            ]

            prompt = (
                f"Startup:\n{json.dumps(self.startup)}\n\n"
                f"Original investor decision:\n"
                f"{json.dumps(self.evaluations[name])}\n\n"
                f"Previous negotiation:\n{json.dumps(history)}\n\n"
                f"Entrepreneur counteroffer:\n{counteroffer}\n\n"
                "Respond realistically. Accept, reject, or counter. "
                "State all proposed terms clearly. "
                "This is a fictional simulation."
            )

            def receive(text):
                write_text(
                    self.negotiation_result,
                    clean_markdown(text)
                )

                self.negotiations.append({
                    "shark": name,
                    "counteroffer": counteroffer,
                    "response": text
                })

                self.report = ""

            self.run_ai(
                self.investor_prompt(name),
                prompt,
                receive,
                850
            )

        button(
            self.content,
            "Submit Counteroffer",
            negotiate,
            True
        ).pack(anchor="w", pady=10)

        button(
            self.content,
            "View Final Report",
            lambda: self.show_page("Report")
        ).pack(anchor="w", pady=8)

    def report_page(self):
        self.clear_page(
            "FINAL INVESTMENT REPORT",
            "Your startup performance and investment assessment."
        )

        report_frame = ctk.CTkFrame(
            self.content,
            fg_color=PANEL,
            corner_radius=12
        )
        report_frame.pack(fill="both", expand=True, pady=12)

        self.report_editor = ctk.CTkTextbox(
            report_frame,
            height=500,
            fg_color=CARD,
            text_color=WHITE,
            font=("Segoe UI", 14),
            wrap="word",
            corner_radius=10
        )
        self.report_editor.pack(
            fill="both",
            expand=True,
            padx=12,
            pady=12
        )

        def display_report(text):
            write_text(
                self.report_editor,
                clean_markdown(text)
            )

        if self.report:
            display_report(self.report)

        def generate():
            if len(self.evaluations) != 5:
                messagebox.showwarning(
                    "Evaluation Required",
                    "Complete all five investor evaluations first."
                )
                return

            system = """
You are a professional startup investment analyst.

Write a comprehensive, readable investment report.

Formatting requirements:
- Use plain text headings.
- Use short paragraphs and numbered points.
- Do not use Markdown tables.
- Do not use Markdown headings, asterisks or code fences.
- Avoid excessively long paragraphs.
- Keep the report professional and easy to read.

Accuracy requirements:
- Preserve exact investor scores and decisions.
- Preserve original investment and equity figures.
- Do not invent financial achievements or startup traction.
- Do not claim a negotiated deal is finalized unless
  the recorded negotiation explicitly confirms agreement.
- All investors and investment offers are fictional.
"""

            prompt = (
                f"Startup:\n{json.dumps(self.startup)}\n\n"
                f"Pitch:\n{self.pitch}\n\n"
                f"Investor Q&A:\n{json.dumps(self.rounds)}\n\n"
                f"Evaluations:\n{json.dumps(self.evaluations)}\n\n"
                f"Negotiations:\n{json.dumps(self.negotiations)}\n\n"
                "Write these sections:\n"
                "1. Executive Summary\n"
                "2. Startup Overview\n"
                "3. Investor-by-Investor Evaluation\n"
                "4. Financial and Valuation Assessment\n"
                "5. Investment Offers and Negotiations\n"
                "6. Strengths and Weaknesses\n"
                "7. Practical Recommendations\n"
                "8. Final Conclusion\n"
                "9. Educational Simulation Disclaimer"
            )

            def receive(text):
                self.report = text
                display_report(text)

            self.run_ai(
                system,
                prompt,
                receive,
                3000
            )

        def save():
            report_text = self.report

            if not report_text:
                messagebox.showwarning(
                    "Empty Report",
                    "Generate the report before saving."
                )
                return

            startup_name = re.sub(
                r"[^A-Za-z0-9_-]",
                "_",
                self.startup.get("name", "Startup")
            )

            path = filedialog.asksaveasfilename(
                title="Save Shark Tank Report",
                defaultextension=".txt",
                initialfile=f"{startup_name}_SharkTank_Report.txt",
                filetypes=[
                    ("Text Document", "*.txt"),
                    ("Markdown Document", "*.md")
                ]
            )

            if not path:
                return

            output = (
                clean_markdown(report_text)
                if path.lower().endswith(".txt")
                else report_text
            )

            Path(path).write_text(
                output,
                encoding="utf-8"
            )

            messagebox.showinfo(
                "Report Saved",
                f"Your report has been saved:\n{path}"
            )

        controls = ctk.CTkFrame(
            self.content,
            fg_color="transparent"
        )
        controls.pack(anchor="w", pady=10)

        button(
            controls,
            "Generate Report",
            generate,
            True
        ).pack(side="left", padx=(0, 10))

        button(
            controls,
            "Save Report",
            save
        ).pack(side="left")

app = SharkTankApp()
app.mainloop()
